<a href="https://colab.research.google.com/github/elayar3/MachineLearning/blob/main/BuyComputer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
data = [
    ["<=30","high","no","fair","no"],
    ["<=30","high","no","excellent","no"],
    ["31-40","high","no","fair","yes"],
    [">40","medium","no","fair","yes"],
    [">40","low","yes","fair","yes"],
    [">40","low","yes","excellent","no"],
    ["31-40","low","yes","excellent","yes"],
    ["<=30","medium","no","fair","no"],
    ["<=30","low","yes","fair","yes"],
    [">40","medium","yes","fair","yes"],
    ["31-40","medium","no","fair","yes"],
    [">40","medium","no","excellent","no"],
    ["<=30","medium","no","fair","no"],
    ["<=30","high","no","excellent","no"],
    ["31-40","medium","no","excellent","yes"],
    ["31-40","high","yes","fair","yes"],
    [">40","medium","no","excellent","no"]
]
df = pd.DataFrame(data, columns=["age","income","student","credit_rating","buys_computer"])
df.head()

,age,income,student,credit_rating,buys_computer
0,<=30,high,no,fair,no
1,<=30,high,no,excellent,no
2,31-40,high,no,fair,yes
3,>40,medium,no,fair,yes
4,>40,low,yes,fair,yes


In [2]:
import math
def entropy(class_col):
    values = class_col.value_counts()
    total = len(class_col)
    return -sum((count/total) * math.log2(count/total) for count in values)

In [3]:
def info_gain(df, attribute, target="buys_computer"):
    total_entropy = entropy(df[target])
    values = df[attribute].unique()

    weighted_entropy = 0
    for v in values:
        subset = df[df[attribute] == v]
        weight = len(subset) / len(df)
        subset_entropy = entropy(subset[target])
        weighted_entropy += weight * subset_entropy

    gain = total_entropy - weighted_entropy
    return gain

In [4]:
for attr in ["income", "student", "credit_rating"]:
    gain = info_gain(df, attr)
    print(f"Gain({attr}) = {gain:.4f}")

Gain(income) = 0.0505
Gain(student) = 0.1562
Gain(credit_rating) = 0.1237


In [5]:
def split_info(df, attribute):
    values = df[attribute].value_counts()
    total = len(df)
    return -sum((c/total) * math.log2(c/total) for c in values)

In [6]:
def gain_ratio(df, attribute, target="buys_computer"):
    ig = info_gain(df, attribute, target)
    si = split_info(df, attribute)
    return ig / si if si != 0 else 0

In [7]:
for attr in ["age", "student"]:
    print(f"Attribute: {attr}")
    print(f"  Information Gain = {info_gain(df, attr):.4f}")
    print(f"  SplitInfo        = {split_info(df, attr):.4f}")
    print(f"  Gain Ratio       = {gain_ratio(df, attr):.4f}")
    print()

Attribute: age
  Information Gain = 0.4151
  SplitInfo        = 1.5799
  Gain Ratio       = 0.2628

Attribute: student
  Information Gain = 0.1562
  SplitInfo        = 0.9367
  Gain Ratio       = 0.1667



In [8]:
import numpy as np

class_counts = df["buys_computer"].value_counts()
total_rows = len(df)

P_yes = class_counts["yes"] / total_rows
P_no  = class_counts["no"]  / total_rows

print("P(yes) =", P_yes)
print("P(no)  =", P_no)

P(yes) = 0.5294117647058824
P(no)  = 0.47058823529411764


In [9]:
def likelihood(attribute, value, class_value):
    subset = df[df["buys_computer"] == class_value]
    return len(subset[subset[attribute] == value]) / len(subset)

In [10]:
X = {
    "age": ">40",
    "income": "low",
    "student": "no",
    "credit_rating": "fair"
}

# Compute likelihoods
P_X_given_yes = np.prod([likelihood(attr, val, "yes") for attr, val in X.items()])
P_X_given_no  = np.prod([likelihood(attr, val, "no")  for attr, val in X.items()])

# Compute posteriors (without denominator)
posterior_yes = P_X_given_yes * P_yes
posterior_no  = P_X_given_no  * P_no

posterior_yes, posterior_no

(np.float64(0.020334059549745823), np.float64(0.007238051470588236))

In [11]:
if posterior_yes > posterior_no:
    print("Predicted Class = YES")
else:
    print("Predicted Class = NO")

Predicted Class = YES
